# Bronze to Silver: Geospatial Risk Intelligence Pipeline

This notebook transforms Bronze-layer Iceberg tables into Silver-layer tables containing
spatially joined, analytically useful datasets using Wherobots Cloud and Apache Sedona.

## Silver tables produced

| Silver Table | Operation | Inputs |
|---|---|---|
| `org_catalog.silver.asset_wildfire_exposure` | Zonal Statistics (raster → vector) | USFS Wildfire Burn Prob / Flame Length + Overture Buildings |
| `org_catalog.silver.asset_flood_exposure` | Zonal Statistics + Temporal Aggregation | MODIS Flood NRT + Overture Buildings |
| `org_catalog.silver.asset_weather_density` | KNN Join + Buffer Aggregation | NOAA SWDI (hail, structure, tvs) + Overture Buildings |
| `org_catalog.silver.asset_enriched` | Conflation Join | All three above |

## Bronze inputs

| Dataset | Catalog Table | Type |
|---|---|---|
| Buildings | `wherobots_open_data.overture_maps_foundation.buildings_building` | Vector |
| Wildfire Burn Probability | `org_catalog.wildfire_risk.burn_probability_conus` | Raster |
| Wildfire Flame Length | `org_catalog.wildfire_risk.conditional_flame_length_conus` | Raster |
| MODIS Flood NRT | `org_catalog.modis.MCDWD_L3_F3_NRT` | Raster |
| NOAA SWDI Hail | `org_catalog.noaa_swdi.hail` | Vector |
| NOAA SWDI Structure | `org_catalog.noaa_swdi.structure` | Vector |
| NOAA SWDI TVS | `org_catalog.noaa_swdi.tvs` | Vector |

## Notebook sequence

```
raw-to-bronze.ipynb  →  bronze-to-silver.ipynb  →  silver-to-gold.ipynb
                             (you are here)
```

## Prerequisites
- Wherobots Cloud runtime with Apache Sedona
- Bronze tables populated (run `raw-to-bronze.ipynb` first)
- `wkls` Python library (`pip install wkls`)

## 0. Configuration & Session Setup

In [7]:
from sedona.spark import *
from pyspark.sql import functions as F
from datetime import datetime, date
import wkls

# ── Pipeline Parameters ──────────────────────────────────────────────────────
# Geographic scope: San Diego, California via wkls
AOI_WKT = wkls.us.ca.sandiego.wkt()
print(f"AOI loaded: San Diego, California ({len(AOI_WKT):,} chars)")

# Temporal windows for before/after analysis
BASELINE_WINDOW_START = "2024-01-01"
BASELINE_WINDOW_END   = "2024-12-31"
EVENT_WINDOW_START    = "2025-01-01"
EVENT_WINDOW_END      = "2025-03-01"

# Wildfire risk classification thresholds (burn probability)
WF_THRESHOLD_EXTREME   = 0.05
WF_THRESHOLD_VERY_HIGH = 0.01
WF_THRESHOLD_HIGH      = 0.002
WF_THRESHOLD_MODERATE  = 0.0005

# ── Table References ─────────────────────────────────────────────────────────
# Bronze / Source tables
BUILDINGS_TABLE   = "wherobots_open_data.overture_maps_foundation.buildings_building"
WILDFIRE_BP_TABLE = "org_catalog.wildfire_risk.burn_probability_conus"
WILDFIRE_FL_TABLE = "org_catalog.wildfire_risk.conditional_flame_length_conus"
FLOOD_TABLE       = "org_catalog.modis.MCDWD_L3_F3_NRT"
SWDI_HAIL_TABLE   = "org_catalog.noaa_swdi.hail"
SWDI_STRUCT_TABLE = "org_catalog.noaa_swdi.structure"
SWDI_TVS_TABLE    = "org_catalog.noaa_swdi.tvs"

# Silver output database (in org_catalog)
SILVER_DB = "silver"

print(f"Baseline window:    {BASELINE_WINDOW_START} → {BASELINE_WINDOW_END}")
print(f"Event window:       {EVENT_WINDOW_START} → {EVENT_WINDOW_END}")

AOI loaded: San Diego, California (137,072 chars)
Baseline window:    2024-01-01 → 2024-12-31
Event window:       2025-01-01 → 2025-03-01
Buffer distances:   0.045° (near), 0.225° (far)


In [2]:
# ── Initialize Wherobots Sedona Session ───────────────────────────────────────
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

# Create the Silver database if it doesn't exist
sedona.sql(f"CREATE DATABASE IF NOT EXISTS org_catalog.{SILVER_DB}")

print("Sedona session initialized ✓")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/23 06:39:01 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.7 ready


Sedona session initialized ✓


## 1. Explore Bronze Tables

Before transforming, verify the Bronze layer is populated and inspect row counts and schema.

In [8]:
# ── Inspect source tables ─────────────────────────────────────────────────────
source_tables = [
    BUILDINGS_TABLE,
    WILDFIRE_BP_TABLE,
    WILDFIRE_FL_TABLE,
    FLOOD_TABLE,
    SWDI_HAIL_TABLE,
    SWDI_STRUCT_TABLE,
    SWDI_TVS_TABLE,
]

for table in source_tables:
    try:
        df = sedona.table(table)
        count = df.count()
        print(f"  ✓ {table:70s} rows: {count:>12,}")
    except Exception as e:
        print(f"  ✗ {table:70s} ERROR: {e}")

  ✓ wherobots_open_data.overture_maps_foundation.buildings_building        rows: 2,639,723,255
  ✓ org_catalog.wildfire_risk.burn_probability_conus                       rows:      970,268
  ✓ org_catalog.wildfire_risk.conditional_flame_length_conus               rows:      970,268
  ✓ org_catalog.modis.MCDWD_L3_F3_NRT                                      rows:      180,500
  ✓ org_catalog.noaa_swdi.hail                                             rows:  288,570,635
  ✓ org_catalog.noaa_swdi.structure                                        rows:  886,022,420
  ✓ org_catalog.noaa_swdi.tvs                                              rows:    1,603,470


In [9]:
# ── Load source tables scoped to AOI ──────────────────────────────────────────
# Buildings: filter by AOI polygon
buildings = sedona.sql(f"""
    SELECT id, geometry, height, num_floors, class, subtype, names.primary AS building_name
    FROM {BUILDINGS_TABLE}
    WHERE ST_Intersects(geometry, ST_GeomFromText('{AOI_WKT}'))
""")
buildings.createOrReplaceTempView("buildings")

# Wildfire rasters: filter tiles overlapping AOI
wildfire_bp = sedona.table(WILDFIRE_BP_TABLE).filter(f"RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))")
wildfire_bp.createOrReplaceTempView("wildfire_bp")

wildfire_fl = sedona.table(WILDFIRE_FL_TABLE).filter(f"RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))")
wildfire_fl.createOrReplaceTempView("wildfire_fl")

# MODIS Flood: filter tiles overlapping AOI
flood = sedona.table(FLOOD_TABLE).filter(f"RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))")
flood.createOrReplaceTempView("flood")

# SWDI tables are loaded per-event-type in section 4 (KNN join)

print(f"Buildings:      {buildings.count():>12,} rows (AOI)")
print(f"Wildfire BP:    {wildfire_bp.count():>12,} tiles")
print(f"Wildfire FL:    {wildfire_fl.count():>12,} tiles")
print(f"Flood:          {flood.count():>12,} tiles")

Buildings:           358,985 rows (AOI)


Wildfire BP:             115 tiles


Wildfire FL:             115 tiles


Flood:                    42 tiles (2024-01-01 → 2025-03-01)


SWDI events:             881 events (2024-01-01 → 2025-03-01)


## 2. Silver Table: `asset_wildfire_exposure`

**Operation**: Zonal Statistics (raster → vector)

For each building footprint, compute the mean and max burn probability from USFS wildfire risk rasters,
plus the mean conditional flame length. Classify each asset into a wildfire risk tier.

> **Sedona Functions**: `RS_ZonalStats(raster, geometry, statType)`, `RS_Intersects`
> 
> **Note**: RS_ZonalStats automatically handles CRS transformation between raster and geometry

In [10]:
# ── 2a. Compute wildfire exposure via zonal stats ────────────────────────────
# Join building footprints with wildfire raster tiles using RS_Intersects,
# then extract burn probability and flame length statistics per asset.
# RS_ZonalStats(raster, geometry, statType) handles CRS alignment automatically.

wildfire_exposure = sedona.sql(f"""
    WITH bp AS (
        SELECT
            b.id AS asset_id,
            b.geometry,
            b.height,
            b.num_floors,
            b.class,
            avg(RS_ZonalStats(w.raster, b.geometry, 'mean')) AS burn_prob_mean,
            max(RS_ZonalStats(w.raster, b.geometry, 'max'))  AS burn_prob_max
        FROM buildings b
        JOIN wildfire_bp w
            ON RS_Intersects(w.raster, b.geometry)
        GROUP BY b.id, b.geometry, b.height, b.num_floors, b.class
    ),
    fl AS (
        SELECT
            b.id AS asset_id,
            avg(RS_ZonalStats(w.raster, b.geometry, 'mean')) AS flame_length_mean
        FROM buildings b
        JOIN wildfire_fl w
            ON RS_Intersects(w.raster, b.geometry)
        GROUP BY b.id
    )
    SELECT
        bp.asset_id,
        'building' AS asset_type,
        bp.geometry,
        bp.height,
        bp.num_floors,
        bp.class,
        bp.burn_prob_mean,
        bp.burn_prob_max,
        COALESCE(fl.flame_length_mean, 0.0) AS flame_length_mean,
        CASE
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_EXTREME}   THEN 'extreme'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_VERY_HIGH} THEN 'very_high'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_HIGH}      THEN 'high'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_MODERATE}  THEN 'moderate'
            ELSE 'low'
        END AS wildfire_risk_class,
        current_timestamp() AS computed_at
    FROM bp
    LEFT JOIN fl ON bp.asset_id = fl.asset_id
""")

wildfire_exposure.cache()
print(f"Wildfire exposure rows: {wildfire_exposure.count():,}")
wildfire_exposure.groupBy("wildfire_risk_class").count().orderBy("count", ascending=False).show()

Wildfire exposure rows: 358,985
+-------------------+------+
|wildfire_risk_class| count|
+-------------------+------+
|                low|351624|
|           moderate|  5184|
|               high|  2116|
|          very_high|    61|
+-------------------+------+



In [11]:
# ── 2b. Write silver.asset_wildfire_exposure to Iceberg ──────────────────────
WILDFIRE_SILVER = f"org_catalog.{SILVER_DB}.asset_wildfire_exposure"

wildfire_exposure.writeTo(WILDFIRE_SILVER).createOrReplace()

row_count = sedona.table(WILDFIRE_SILVER).count()
print(f"✓ Wrote {row_count:,} rows to {WILDFIRE_SILVER}")

✓ Wrote 358,985 rows to org_catalog.silver.asset_wildfire_exposure


## 3. Silver Table: `asset_flood_exposure`

**Operation**: Zonal Statistics + Temporal Aggregation (raster → vector)

For each building, aggregate MODIS NRT flood signals over the observation window:
max flood class code, number of distinct flood acquisition dates, and duration of flooding.

> **Sedona Functions**: `RS_ZonalStats(raster, geometry, statType)`, `RS_Intersects`
>
> **Source**: `org_catalog.modis.MCDWD_L3_F3_NRT` — raster column `raster`, temporal column `acq_date`
>
> **Note**: MCDWD pixel values are categorical (1=water, 2=flood, 3=no-data), not continuous depth.
> `flood_event_count` is the primary meaningful continuous metric for downstream scoring.

In [12]:
# ── 3a. Compute flood exposure via zonal stats + temporal aggregation ─────────
# For each building intersecting a MODIS flood tile within the observation window,
# compute the max flood class, count of distinct flood events, and flood duration.
# NOTE: MCDWD pixel values are categorical (1=water, 2=flood, 3=no-data),
# so flood_max_class is the highest category observed, not a continuous depth.

flood_exposure = sedona.sql(f"""
    SELECT
        b.id                                                       AS asset_id,
        'building'                                                 AS asset_type,
        b.geometry,
        MAX(RS_ZonalStats(f.raster, b.geometry, 'max'))            AS flood_max_class,
        COUNT(DISTINCT f.acq_date)                                 AS flood_event_count,
        DATEDIFF(MAX(f.acq_date), MIN(f.acq_date))                AS flood_duration_days,
        DATE('{BASELINE_WINDOW_START}')                            AS observation_window_start,
        DATE('{EVENT_WINDOW_END}')                                 AS observation_window_end,
        current_timestamp()                                        AS computed_at
    FROM buildings b
    JOIN flood f
        ON RS_Intersects(f.raster, b.geometry)
    WHERE f.acq_date BETWEEN DATE('{BASELINE_WINDOW_START}') AND DATE('{EVENT_WINDOW_END}')
    GROUP BY b.id, b.geometry
""")

flood_exposure.cache()
print(f"Flood exposure rows: {flood_exposure.count():,}")
flood_exposure.show(5, truncate=False)

Flood exposure rows: 358,985
+------------------------------------+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+-----------------+-------------------+------------------------+----------------------+-------------------------+
|asset_id                            |asset_type|geometry                                                                                                                                                                                                                                   |flood_max_extent|flood_event_count|flood_duration_days|observation_window_start|observation_window_end|computed_at              |
+------------------------------------+----------+--------------------------------------------------------------------------------------------

In [13]:
# ── 3b. Write silver.asset_flood_exposure to Iceberg ─────────────────────────
FLOOD_SILVER = f"org_catalog.{SILVER_DB}.asset_flood_exposure"

flood_exposure.writeTo(FLOOD_SILVER).createOrReplace()

row_count = sedona.table(FLOOD_SILVER).count()
print(f"✓ Wrote {row_count:,} rows to {FLOOD_SILVER}")

✓ Wrote 358,985 rows to org_catalog.silver.asset_flood_exposure


## 4. Silver Table: `asset_weather_density`

**Operation**: KNN Join with search radius + Spheroidal Distance (vector → vector)

For each building, find the K nearest NOAA SWDI severe weather events **per event type**
(hail, structure, TVS) using WherobotsDB's KNN join with `search_radius` to cap the search
at 25 km. Uses `use_sphere = true` so both the KNN search and the radius are in meters.

> **WherobotsDB**: `ST_KNN(R, S, k, use_sphere, search_radius)` — 5th param limits search distance
>
> **Sources**: `org_catalog.noaa_swdi.hail`, `.structure`, `.tvs` — queried separately per event type

In [20]:
# ── 4a. KNN join per SWDI event type → append to Iceberg ─────────────────────
# Use ST_KNN with search_radius (Wherobots extension) to limit neighbor search
# to 25 km. use_sphere=TRUE means both the KNN distance and search_radius are
# in meters. Pre-filter each SWDI table to AOI + temporal window first.

K_NEAREST      = 10
SEARCH_RADIUS  = 25000    # 25 km in meters (search_radius param)
BUFFER_NEAR_M  = 5000     # 5 km threshold for aggregation
BUFFER_FAR_M   = 25000    # 25 km threshold for aggregation

KNN_STAGING = f"org_catalog.{SILVER_DB}.weather_knn_staging"

swdi_sources = {
    "hail":      (SWDI_HAIL_TABLE,   "SEVPROB",     "MAXSIZE"),
    "structure": (SWDI_STRUCT_TABLE,  "MAX_REFLECT", "VIL"),
    "tvs":       (SWDI_TVS_TABLE,    "MXDV",        "MAX_SHEAR"),
}

first = True
for event_type, (table, severity_col, magnitude_col) in swdi_sources.items():
    # Step 1: pre-filter to AOI + temporal window
    view_name = f"swdi_{event_type}_aoi"
    sedona.sql(f"""
        SELECT geometry, ZTIME, {severity_col}, {magnitude_col}
        FROM {table}
        WHERE ZTIME BETWEEN TIMESTAMP('{BASELINE_WINDOW_START}') AND TIMESTAMP('{EVENT_WINDOW_END}')
          AND ST_Intersects(geometry, ST_GeomFromText('{AOI_WKT}'))
    """).createOrReplaceTempView(view_name)
    cnt = sedona.sql(f"SELECT COUNT(*) AS c FROM {view_name}").collect()[0]["c"]
    print(f"  {event_type}: {cnt:,} events in AOI")

    # Step 2: KNN join with use_sphere=TRUE and search_radius=25km
    df = sedona.sql(f"""
        SELECT
            b.id                                              AS asset_id,
            b.geometry                                        AS asset_geometry,
            '{event_type}'                                    AS event_type,
            e.ZTIME                                           AS event_time,
            e.{severity_col}                                  AS severity,
            e.{magnitude_col}                                 AS magnitude,
            ST_DistanceSpheroid(b.geometry, e.geometry)        AS distance_m
        FROM buildings b
        JOIN {view_name} e
            ON ST_KNN(b.geometry, e.geometry, {K_NEAREST}, true, {SEARCH_RADIUS})
    """)

    # Step 3: write to Iceberg — createOrReplace first, append after
    if first:
        df.writeTo(KNN_STAGING).createOrReplace()
        first = False
    else:
        df.writeTo(KNN_STAGING).append()
    print(f"  ✓ {event_type} written to {KNN_STAGING}")

total = sedona.table(KNN_STAGING).count()
print(f"\nTotal staging rows: {total:,}")
sedona.table(KNN_STAGING).show(10, truncate=False)

  hail: 3 events in AOI


26/03/23 07:08:00 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. This may cause the KNN join to return incorrect results. Consider materializing the object side before the join to prevent filter pushdown.


  ✓ hail written to org_catalog.silver.weather_knn_staging


  structure: 878 events in AOI


26/03/23 07:08:20 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. This may cause the KNN join to return incorrect results. Consider materializing the object side before the join to prevent filter pushdown.


  ✓ structure written to org_catalog.silver.weather_knn_staging


  tvs: 0 events in AOI


26/03/23 07:09:09 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. This may cause the KNN join to return incorrect results. Consider materializing the object side before the join to prevent filter pushdown.


  ✓ tvs written to org_catalog.silver.weather_knn_staging

Total staging rows: 4,494,050
+------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+-------------------+--------+---------+------------------+
|asset_id                            |asset_geometry                                                                                                                                                                          |event_type|event_time         |severity|magnitude|distance_m        |
+------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+-------------------+--------+---------+------------------+
|1a93280f-2d03-49dc-b716-8a05c83

In [21]:
# ── 4b. Aggregate KNN staging → asset_weather_density ────────────────────────
# Read from the Iceberg staging table (no in-memory cache needed).

WEATHER_SILVER = f"org_catalog.{SILVER_DB}.asset_weather_density"

weather_density = sedona.sql(f"""
    SELECT
        asset_id,
        'building'                                                 AS asset_type,
        asset_geometry                                             AS geometry,

        SUM(CASE WHEN distance_m <= {BUFFER_NEAR_M} THEN 1 ELSE 0 END)   AS event_count_5km,
        SUM(CASE WHEN distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END)    AS event_count_25km,

        MIN(distance_m)                                            AS nearest_event_dist_m,

        MIN(CASE WHEN event_type = 'hail'      THEN distance_m END) AS nearest_hail_m,
        MIN(CASE WHEN event_type = 'structure'  THEN distance_m END) AS nearest_structure_m,
        MIN(CASE WHEN event_type = 'tvs'        THEN distance_m END) AS nearest_tvs_m,

        SUM(CASE WHEN event_type = 'hail'      AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS hail_count_25km,
        SUM(CASE WHEN event_type = 'structure'  AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS structure_count_25km,
        SUM(CASE WHEN event_type = 'tvs'        AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS tvs_count_25km,

        -- Per-event-type max severity (units differ: SEVPROB 0-100, MAX_REFLECT dBZ, MXDV knots)
        MAX(CASE WHEN event_type = 'hail'      THEN severity END)  AS max_hail_sevprob,
        MAX(CASE WHEN event_type = 'structure'  THEN severity END) AS max_structure_reflectivity,
        MAX(CASE WHEN event_type = 'tvs'        THEN severity END) AS max_tvs_delta_v,

        DATE('{BASELINE_WINDOW_START}')                            AS observation_window_start,
        DATE('{EVENT_WINDOW_END}')                                 AS observation_window_end,
        current_timestamp()                                        AS computed_at

    FROM {KNN_STAGING}
    GROUP BY asset_id, asset_geometry
""")

weather_density.writeTo(WEATHER_SILVER).createOrReplace()
row_count = sedona.table(WEATHER_SILVER).count()
print(f"✓ Wrote {row_count:,} rows to {WEATHER_SILVER}")

✓ Wrote 358,985 rows to org_catalog.silver.asset_weather_density


## 5. Silver Table: `asset_enriched`

**Operation**: Conflation Join — Merge all three hazard exposure layers onto the base asset table

This is the unified Silver table that Gold will consume. Each row is an asset with all hazard signals attached. Uses `LEFT JOIN` so assets with no exposure in a given hazard still appear (with nulls).

In [ ]:
# ── 5a. Conflate all exposure layers onto base assets ────────────────────────
# LEFT JOIN ensures every building in scope appears even if it has no
# wildfire/flood/weather exposure — those columns will be null.

WILDFIRE_SILVER = f"org_catalog.{SILVER_DB}.asset_wildfire_exposure"
FLOOD_SILVER    = f"org_catalog.{SILVER_DB}.asset_flood_exposure"
WEATHER_SILVER  = f"org_catalog.{SILVER_DB}.asset_weather_density"

asset_enriched = sedona.sql(f"""
    SELECT
        b.id                               AS asset_id,
        'building'                         AS asset_type,
        b.geometry,
        b.class                            AS building_class,
        b.height,
        b.num_floors,

        -- Wildfire exposure
        w.burn_prob_mean,
        w.burn_prob_max,
        w.flame_length_mean,
        w.wildfire_risk_class,

        -- Flood exposure
        f.flood_max_class,
        f.flood_event_count,
        f.flood_duration_days,

        -- Severe weather density (KNN-derived)
        s.event_count_5km,
        s.event_count_25km,
        s.nearest_event_dist_m,
        s.nearest_hail_m,
        s.nearest_structure_m,
        s.nearest_tvs_m,
        s.hail_count_25km,
        s.structure_count_25km,
        s.tvs_count_25km,
        s.max_hail_sevprob,
        s.max_structure_reflectivity,
        s.max_tvs_delta_v,

        -- Data coverage flags (null = no data, not zero risk)
        CASE WHEN w.asset_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_wildfire_data,
        CASE WHEN f.asset_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_flood_data,
        CASE WHEN s.asset_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_weather_data,

        -- Temporal windows
        DATE('{BASELINE_WINDOW_START}')    AS baseline_window_start,
        DATE('{BASELINE_WINDOW_END}')      AS baseline_window_end,
        DATE('{EVENT_WINDOW_START}')       AS event_window_start,
        DATE('{EVENT_WINDOW_END}')         AS event_window_end,

        current_timestamp()                AS computed_at

    FROM buildings b
    LEFT JOIN {WILDFIRE_SILVER} w
        ON b.id = w.asset_id
    LEFT JOIN {FLOOD_SILVER} f
        ON b.id = f.asset_id
    LEFT JOIN {WEATHER_SILVER} s
        ON b.id = s.asset_id
""")

asset_enriched.cache()
print(f"Asset enriched rows: {asset_enriched.count():,}")
asset_enriched.printSchema()

Asset enriched rows: 358,985
root
 |-- asset_id: string (nullable = true)
 |-- asset_type: string (nullable = false)
 |-- geometry: geometry (nullable = true)
 |-- building_class: string (nullable = true)
 |-- height: double (nullable = true)
 |-- num_floors: integer (nullable = true)
 |-- burn_prob_mean: double (nullable = true)
 |-- burn_prob_max: double (nullable = true)
 |-- flame_length_mean: double (nullable = true)
 |-- wildfire_risk_class: string (nullable = true)
 |-- flood_max_extent: double (nullable = true)
 |-- flood_event_count: long (nullable = true)
 |-- flood_duration_days: integer (nullable = true)
 |-- event_count_5km: long (nullable = true)
 |-- event_count_25km: long (nullable = true)
 |-- nearest_event_dist_m: double (nullable = true)
 |-- nearest_hail_m: double (nullable = true)
 |-- nearest_structure_m: double (nullable = true)
 |-- nearest_tvs_m: double (nullable = true)
 |-- hail_count_25km: long (nullable = true)
 |-- structure_count_25km: long (nullable = tr

In [ ]:
# ── 5b. Quick data quality check ──────────────────────────────────────────────
# Show coverage: what % of assets have non-null values for each hazard layer.

total = asset_enriched.count()
wf_coverage = asset_enriched.filter(F.col("has_wildfire_data")).count()
fl_coverage = asset_enriched.filter(F.col("has_flood_data")).count()
sw_coverage = asset_enriched.filter(F.col("has_weather_data")).count()

print(f"Total assets:                  {total:>10,}")
print(f"Wildfire exposure coverage:    {wf_coverage:>10,}  ({wf_coverage/total*100:.1f}%)")
print(f"Flood exposure coverage:       {fl_coverage:>10,}  ({fl_coverage/total*100:.1f}%)")
print(f"Severe weather coverage:       {sw_coverage:>10,}  ({sw_coverage/total*100:.1f}%)")
print()

# Distribution of wildfire risk classes
print("Wildfire risk class distribution:")
asset_enriched.groupBy("wildfire_risk_class").count().orderBy("count", ascending=False).show()

Total assets:                     358,985
Wildfire exposure coverage:        85,375  (23.8%)
Flood exposure coverage:              476  (0.1%)
Severe weather coverage:          358,985  (100.0%)

Wildfire risk class distribution:
+-------------------+------+
|wildfire_risk_class| count|
+-------------------+------+
|                low|351624|
|           moderate|  5184|
|               high|  2116|
|          very_high|    61|
+-------------------+------+



In [24]:
# ── 5c. Write silver.asset_enriched to Iceberg ───────────────────────────────
ENRICHED_SILVER = f"org_catalog.{SILVER_DB}.asset_enriched"

asset_enriched.writeTo(ENRICHED_SILVER).createOrReplace()

row_count = sedona.table(ENRICHED_SILVER).count()
print(f"✓ Wrote {row_count:,} rows to {ENRICHED_SILVER}")

✓ Wrote 358,985 rows to org_catalog.silver.asset_enriched


## 6. Verify Silver Layer

Final verification: list all Silver tables and confirm row counts.

In [25]:
# ── Verify all Silver tables ──────────────────────────────────────────────────
silver_tables = [
    "asset_wildfire_exposure",
    "asset_flood_exposure",
    "asset_weather_density",
    "asset_enriched",
]

print("Silver Layer Summary")
print("=" * 72)
for table in silver_tables:
    fqn = f"org_catalog.{SILVER_DB}.{table}"
    df = sedona.table(fqn)
    count = df.count()
    cols = len(df.columns)
    print(f"  {fqn:55s}  {count:>10,} rows  {cols:>3} cols")
print("=" * 72)
print("\n✓ Bronze → Silver pipeline complete.")
print(f"  Next step: run silver-to-gold.ipynb to produce industry-specific Gold tables.")

Silver Layer Summary
  org_catalog.silver.asset_wildfire_exposure                  358,985 rows   11 cols
  org_catalog.silver.asset_flood_exposure                     358,985 rows    9 cols
  org_catalog.silver.asset_weather_density                    358,985 rows   16 cols
  org_catalog.silver.asset_enriched                           358,985 rows   28 cols

✓ Bronze → Silver pipeline complete.
  Next step: run silver-to-gold.ipynb to produce industry-specific Gold tables.
